In [ ]:
import mpltex
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.cbook import get_sample_data
import pandas as pd
import numpy as np
import random, itertools, math, re, os
import csv
from collections import defaultdict
os.environ["PATH"] = "/Library/TeX/texbin:" + os.environ["PATH"]

plt.rcParams['pgf.texsystem'] = 'pdflatex'
plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

In [ ]:
import subprocess
print("PATH:", os.environ.get("PATH", ""))
print("latex:", subprocess.run(["which", "latex"], capture_output=True, text=True).stdout)

In [ ]:
df = pd.read_csv("../../data/success_rate/converge_rate_list.csv")

In [ ]:
for i in df['algorithm']:
    print(i, df[df['algorithm'] == i]['convergence_rate'].values[0])

In [ ]:
@mpltex.acs_decorator
def plot_converge_rate():
    wu = np.arange(50, 180, 10)
    fig, ax = plt.subplots(figsize=(4, 3))
    for algo in df['algorithm']:
        converge_rate = eval(df[df['algorithm'] == algo]['convergence_rate'].values[0])
        if algo == 'AA-SD':
            ax.plot(wu, converge_rate[5:18], label=algo, linewidth=1, color='black', marker='v', markersize=2)
        else:
            ax.plot(wu, converge_rate[5:18], label=algo)

    ax.set_xlabel(r'$N_{\text{warmup}}$', fontsize=10)
    ax.set_ylabel('success rate', fontsize=10)
    ax.grid(False)
    ax.legend(loc='right', fontsize=10, frameon=False)
    plt.xlim(49, 176)
    plt.ylim(-0.02, 1.05)
    plt.xticks(np.arange(50, 180, 20))
    plt.yticks(np.arange(0.0, 1.1, 0.2))
    plt.tight_layout(pad=0.1)
    plt.savefig('rate.svg')
    plt.savefig('rate.pdf')
    plt.savefig('rate.png')
    plt.show()
plot_converge_rate()

In [ ]:
# warmup作为行索引，算法作为列索引（60-150）
# 突出显示每个warmup下的最大成功率

warmups_array = np.arange(0, 210, 10)  # 0, 10, 20, ..., 200
idx_60 = np.where(warmups_array == 60)[0][0]
idx_150 = np.where(warmups_array == 150)[0][0]

warmups_range = warmups_array[idx_60:idx_150+1]
print(f"Warmups (行索引): {warmups_range}")

# 重新整理数据，按照 AA/NA/OA 分组，每组有 ETD/SD/SIS
data_by_prefix = {}
for algo in df['algorithm'].values:
    rates_str = df[df['algorithm'] == algo]['convergence_rate'].values[0]
    rates = eval(rates_str)
    prefix, method = algo.split('-')
    
    if prefix not in data_by_prefix:
        data_by_prefix[prefix] = {}
    data_by_prefix[prefix][method] = rates[idx_60:idx_150+1]

# 生成LaTeX表格：warmup为行，AA/NA/OA为列，并突出显示每行最大值
latex_content = r"""\documentclass{article}
\usepackage{booktabs}
\usepackage{multirow}
\usepackage{graphicx}
\usepackage{caption}

\begin{document}

\begin{table}[htbp]
\centering
\caption{Success rates for convergence under different warmups (60-150).}
\label{tab:convergence_rate}
\resizebox{\textwidth}{!}{
\begin{tabular}{c ccc ccc ccc}
\toprule
\multirow{2}{*}{$N_{\mathrm{warmup}}$} & \multicolumn{3}{c}{\textbf{AA}} & \multicolumn{3}{c}{\textbf{NA}} & \multicolumn{3}{c}{\textbf{OA}} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10}
 & ETD & SD & SIS & ETD & SD & SIS & ETD & SD & SIS \\
\midrule
"""

# 按warmup添加行，并找出每行最大值
for i, warmup in enumerate(warmups_range):
    row_values = []
    
    # 收集所有9个值
    for prefix in ['AA', 'NA', 'OA']:
        for method in ['ETD', 'SD', 'SIS']:
            val = data_by_prefix[prefix][method][i]
            row_values.append(val)
    
    # 找出最大值
    max_val = max(row_values)
    
    # 生成行，最大值用\bf加粗
    row_strings = [f"{int(warmup)}"]
    val_idx = 0
    for prefix in ['AA', 'NA', 'OA']:
        for method in ['ETD', 'SD', 'SIS']:
            val = row_values[val_idx]
            if val == max_val and val > 0:  # 只有非零值才加粗
                row_strings.append(f"\\bf{{{val:.2f}}}")
            else:
                row_strings.append(f"{val:.2f}")
            val_idx += 1
    
    latex_content += " & ".join(row_strings) + r" \\" + "\n"

latex_content += r"""\bottomrule
\end{tabular}
}
\end{table}

\end{document}
"""

print("\n" + "="*80)
print("生成的LaTeX表格（warmup为行，60-150，突出显示最大值）:")
print("="*80)
print(latex_content)
